In [1]:
"""
MODIFIED PSD-BASED GENERATOR FOR RUNNING
Adapted from your cycling generator with running-specific parameters
"""

import numpy as np
import pandas as pd
from scipy.signal import welch, butter, filtfilt

def generate_synthetic_running_accel(stat, duration_sec=240, seed=None,
                                     cadence_var=0.20,      # Higher variation for running
                                     amp_var=0.8,           # Higher amplitude variation
                                     bias_scale=0.03,       # Slightly more bias
                                     noise_scale=1.2,       # More noise (foot strikes)
                                     harmonic_ratio=0.25,   # Less harmonics
                                     bump_density=15,       # More frequent impacts
                                     strike_sharpness=0.7): # NEW: Foot strike sharpness
    """
    Generate synthetic running accelerometer data

    KEY CHANGES FROM CYCLING:
    1. Cadence: 2.7-3.0 Hz (vs 1.5-2.0 Hz cycling)
    2. Higher vertical impact
    3. Sharper transients (foot strikes)
    4. More noise/variation
    5. Higher amplitude

    Parameters
    ----------
    stat : dict
        Statistics from real running data
        Must contain: fs, mean_xyz, std_xyz, psd_xyz, freqs
    duration_sec : float
        Duration in seconds
    cadence_var : float
        Running cadence variation (0.20 = ±20%)
    amp_var : float
        Amplitude variation (higher for running)
    strike_sharpness : float
        NEW: Controls foot strike sharpness (0-1)
    """

    if seed is not None:
        np.random.seed(seed)

    fs = stat['fs']
    N = int(duration_sec * fs)
    t = np.linspace(0, duration_sec, N, endpoint=False)

    # RUNNING-SPECIFIC PARAMETERS
    # ============================

    # 1. CADENCE: Running is faster (2.7-3.0 Hz vs 1.5-2.0 Hz)
    base_cadence_running = 2.85  # Hz (171 SPM - typical running)
    cadence = base_cadence_running * (1 + cadence_var * (np.random.rand() - 0.5))

    print(f"   Generating running data:")
    print(f"     Cadence: {cadence:.2f} Hz ({cadence*60:.0f} SPM)")
    print(f"     Duration: {duration_sec}s")
    print(f"     Samples: {N}")

    # 2. VERTICAL COMPONENT: Much stronger for running
    # Foot strike creates sharp vertical acceleration
    vertical_base = np.sin(2 * np.pi * cadence * t)

    # Add foot strike sharpness (percussive impact)
    foot_strike_component = np.zeros_like(t)
    stride_period = 1.0 / cadence
    n_strides = int(duration_sec / stride_period)

    for stride in range(n_strides):
        strike_time = stride * stride_period
        strike_idx = int(strike_time * fs)
        if strike_idx < N:
            # Sharper transient for foot strike
            decay_samples = int(0.05 * fs)  # 50ms decay
            for i in range(min(decay_samples, N - strike_idx)):
                foot_strike_component[strike_idx + i] += \
                    strike_sharpness * np.exp(-i / (decay_samples * 0.3))

    # 3. GENERATE PER-AXIS SIGNALS
    xyz = {}

    for axis in ['x', 'y', 'z']:
        # Base sinusoidal component
        base = np.sin(2 * np.pi * cadence * t + np.random.uniform(0, 2*np.pi))

        # Z-axis (vertical) gets strong foot strike component
        if axis == 'z':
            base = base + 2.0 * foot_strike_component  # Strong vertical impact
        # Y-axis (forward) gets moderate component
        elif axis == 'y':
            base = base + 0.5 * foot_strike_component
        # X-axis (lateral) gets weak component
        else:
            base = base + 0.3 * foot_strike_component

        # Add harmonics (less than cycling - running is more percussive)
        for h in [2, 3]:
            harmonic_amp = harmonic_ratio / h
            base += harmonic_amp * np.sin(2 * np.pi * h * cadence * t +
                                          np.random.uniform(0, 2*np.pi))

        # Add noise (more than cycling due to impact variability)
        noise = noise_scale * np.random.randn(N)

        # Apply low-pass filter (running has sharper transients than cycling)
        # Use higher cutoff frequency
        cutoff = 15.0  # Hz (vs 10 Hz for cycling)
        # sos = butter(4, cutoff, fs=fs, output='sos')
        # signal = filtfilt(sos, base + noise)
        # CORRECT (BA format with filtfilt)
        b, a = butter(4, cutoff, fs=fs, btype='low')
        signal = filtfilt(b, a, base + noise)  # ✅

        # Scale to match real data statistics
        real_mean = stat[f'mean_{axis}']
        real_std = stat[f'std_{axis}']

        # Normalize and scale
        signal = signal - signal.mean()
        signal = signal / (signal.std() + 1e-10)
        signal = signal * real_std * (1 + amp_var * (np.random.rand() - 0.5))
        signal = signal + real_mean + bias_scale * np.random.randn()

        xyz[axis] = signal

    # 4. Apply cross-axis correlation (if available)
    if 'corr_matrix' in stat:
        # Use Cholesky decomposition to impose correlation
        corr_matrix = stat['corr_matrix']
        L = np.linalg.cholesky(corr_matrix)

        data_matrix = np.array([xyz['x'], xyz['y'], xyz['z']])

        # Standardize
        means = data_matrix.mean(axis=1, keepdims=True)
        stds = data_matrix.std(axis=1, keepdims=True)
        data_standardized = (data_matrix - means) / (stds + 1e-10)

        # Apply correlation
        data_correlated = L @ data_standardized

        # Destandardize
        data_final = data_correlated * stds + means

        xyz['x'] = data_final[0]
        xyz['y'] = data_final[1]
        xyz['z'] = data_final[2]

    # 5. Create DataFrame
    df_synth = pd.DataFrame({
        'seconds_elapsed': t,
        'x': xyz['x'],
        'y': xyz['y'],
        'z': xyz['z']
    })

    return df_synth


def extract_running_statistics(real_running_data):
    """
    Extract statistics from real running data

    Parameters
    ----------
    real_running_data : DataFrame
        Must have columns: seconds_elapsed, x, y, z (or acc_x, acc_y, acc_z)

    Returns
    -------
    stat : dict
        Statistics dictionary for generation
    """

    # Handle different column naming
    if 'acc_x' in real_running_data.columns:
        x_col, y_col, z_col = 'acc_x', 'acc_y', 'acc_z'
    else:
        x_col, y_col, z_col = 'x', 'y', 'z'

    # Compute sampling rate
    t = real_running_data['time_seconds'].values
    dt = np.diff(t).mean()
    fs = 1.0 / dt

    # Extract signals
    x = real_running_data[x_col].values
    y = real_running_data[y_col].values
    z = real_running_data[z_col].values

    # Basic statistics
    stat = {
        'fs': fs,
        'mean_x': x.mean(),
        'mean_y': y.mean(),
        'mean_z': z.mean(),
        'std_x': x.std(),
        'std_y': y.std(),
        'std_z': z.std(),
    }

    # Correlation matrix
    data_matrix = np.array([x, y, z])
    stat['corr_matrix'] = np.corrcoef(data_matrix)

    # PSD for each axis (optional, for validation)
    for signal, axis in zip([x, y, z], ['x', 'y', 'z']):
        freqs, psd = welch(signal, fs=fs, nperseg=min(2048, len(signal)//4))
        stat[f'freqs_{axis}'] = freqs
        stat[f'psd_{axis}'] = psd

    print(f"Extracted statistics:")
    print(f"  Sampling rate: {fs:.1f} Hz")
    print(f"  Mean: x={stat['mean_x']:.3f}, y={stat['mean_y']:.3f}, z={stat['mean_z']:.3f}")
    print(f"  Std: x={stat['std_x']:.3f}, y={stat['std_y']:.3f}, z={stat['std_z']:.3f}")

    return stat


# VALIDATION FUNCTIONS
# ====================

def validate_running_synthetic(real_data, synth_data, stat):
    """
    Validate synthetic running data against real

    Checks:
    1. Cadence detection
    2. PSD comparison
    3. Amplitude distribution
    4. Foot strike characteristics
    """

    from scipy.signal import find_peaks, welch
    import matplotlib.pyplot as plt

    fs = stat['fs']

    print("\n" + "="*70)
    print("RUNNING SYNTHETIC VALIDATION")
    print("="*70)

    # 1. CADENCE DETECTION
    print("\n1. Cadence Detection:")

    for df, label in [(real_data, 'Real'), (synth_data, 'Synthetic')]:
        # Use z-axis (vertical) for cadence
        z = df['z'].values if 'z' in df.columns else df['acc_z'].values

        # Find peaks (foot strikes)
        peaks, _ = find_peaks(z, distance=int(fs*0.3), prominence=0.5)

        if len(peaks) > 1:
            cadence_hz = len(peaks) / (len(z) / fs)
            cadence_spm = cadence_hz * 60
            print(f"   {label}: {cadence_hz:.2f} Hz ({cadence_spm:.0f} SPM)")
        else:
            print(f"   {label}: Could not detect cadence")

    # 2. PSD COMPARISON
    print("\n2. Power Spectral Density:")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for idx, axis in enumerate(['x', 'y', 'z']):
        real_col = axis if axis in real_data.columns else f'acc_{axis}'
        synth_col = axis if axis in synth_data.columns else f'acc_{axis}'

        real_signal = real_data[real_col].values
        synth_signal = synth_data[synth_col].values

        freqs_real, psd_real = welch(real_signal, fs=fs, nperseg=min(2048, len(real_signal)//4))
        freqs_synth, psd_synth = welch(synth_signal, fs=fs, nperseg=min(2048, len(synth_signal)//4))

        axes[idx].semilogy(freqs_real, psd_real, label='Real', linewidth=2)
        axes[idx].semilogy(freqs_synth, psd_synth, label='Synthetic', linewidth=2, linestyle='--')
        axes[idx].set_xlim([0, 10])
        axes[idx].axvline(2.7, color='red', linestyle=':', label='Running cadence zone')
        axes[idx].axvline(3.0, color='red', linestyle=':')
        axes[idx].set_xlabel('Frequency (Hz)')
        axes[idx].set_ylabel('PSD')
        axes[idx].set_title(f'Axis: {axis}')
        axes[idx].legend()
        axes[idx].grid(True, alpha=0.3)

    plt.suptitle('PSD Comparison: Running', fontweight='bold')
    plt.tight_layout()
    plt.savefig('running_psd_validation.png', dpi=150)
    print("   ✓ Saved: running_psd_validation.png")
    plt.close()

    # 3. AMPLITUDE COMPARISON
    print("\n3. Amplitude Statistics:")

    for axis in ['x', 'y', 'z']:
        real_col = axis if axis in real_data.columns else f'acc_{axis}'
        synth_col = axis if axis in synth_data.columns else f'acc_{axis}'

        real_std = real_data[real_col].std()
        synth_std = synth_data[synth_col].std()

        diff_pct = abs(real_std - synth_std) / real_std * 100
        status = "✅" if diff_pct < 30 else "⚠️"

        print(f"   {axis}: Real σ={real_std:.3f}, Synth σ={synth_std:.3f}, "
              f"Diff={diff_pct:.1f}% {status}")

    print("\n✅ Validation complete!")


# USAGE EXAMPLE
# =============


# Step 1: Load your running data
running_df = pd.read_csv('/content/drive/MyDrive/CS5103_IITH_PMA_project/Ankit/Running/merged_sensors_100hz.csv')

# Step 2: Extract statistics
stat = extract_running_statistics(running_df)

# Step 3: Generate synthetic running data
synth_running = generate_synthetic_running_accel(
    stat,
    duration_sec=240,
    cadence_var=0.20,
    amp_var=0.8,
    strike_sharpness=0.7
)

# Step 4: Validate
validate_running_synthetic(running_df, synth_running, stat)

# Step 5: Save
synth_running.to_csv('synthetic_running.csv', index=False)


Extracted statistics:
  Sampling rate: 100.0 Hz
  Mean: x=-0.035, y=-0.070, z=0.001
  Std: x=1.480, y=1.318, z=1.337
   Generating running data:
     Cadence: 2.82 Hz (169 SPM)
     Duration: 240s
     Samples: 23999

RUNNING SYNTHETIC VALIDATION

1. Cadence Detection:
   Real: 0.84 Hz (51 SPM)
   Synthetic: 2.73 Hz (164 SPM)

2. Power Spectral Density:
   ✓ Saved: running_psd_validation.png

3. Amplitude Statistics:
   x: Real σ=1.480, Synth σ=1.650, Diff=11.4% ✅
   y: Real σ=1.318, Synth σ=1.083, Diff=17.8% ✅
   z: Real σ=1.337, Synth σ=1.001, Diff=25.1% ✅

✅ Validation complete!
